In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd
import os
import re

from typing import Tuple

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("direcciones.csv", sep=";", encoding="latin1")

In [ ]:
df

In [ ]:
def hora_dec(num:str)-> float:

    pattern = re.compile(r"(\d+)°(\d+)'([\d\.]+)\'\' ([NSWE]+)")

    coords = re.fullmatch(pattern, num)
    coords_grados = float(coords.group(1))
    coords_minutos = float(coords.group(2))
    coords_segundos = float(coords.group(3))
    coords_direccion = coords.group(4)

    coords_decimal =  coords_grados + (coords_minutos / 60) + (coords_segundos / 3600)
    if  coords_direccion in ["S", "W"]:
        coords_decimal =  coords_decimal

    return coords_decimal


def carga_callejero() -> pd.DataFrame:
    """ Función que carga el callejero de Madrid, lo procesa y devuelve
    un DataFrame con los datos procesados
    
    Args: None
    Returns:
        DataFrame: dataframe con los datos del callejero procesados.
    Raises:
        FileNotFoundError si el fichero csv con las direcciones no existe
    """
    file = "direcciones.csv"

    # if not os.path.exists(file):
    #     raise FileNotFoundError(f"Fichero {file} no encontrado en el PATH")

    columnas = ["VIA_CLASE", "VIA_PAR", "VIA_NOMBRE","NUMERO",  "LATITUD", "LONGITUD"]

    df = pd.read_csv(file, sep=";", usecols=columnas,encoding=  "latin1")

    df.head()

    df["VIA_COMPLETA"] = (
                        df["VIA_CLASE"].str.strip().str.title() +
                        " " +
                        df["VIA_PAR"].str.strip().str.lower() +
                        " " +
                        df["VIA_NOMBRE"].str.strip().str.title() +
                        ", " +
                        df["NUMERO"].apply(str).str.strip()
                        )

    df["LONGITUD"]= df["LONGITUD"].apply(hora_dec)
    df["LATITUD"]= df["LATITUD"].apply(hora_dec)

    return df
    



In [ ]:
def carga_grafo() -> nx.MultiDiGraph:
    """ Función que recupera el quiver de calles de Madrid de OpenStreetMap.
    Args: None
    Returns:
        nx.MultiDiGraph: Quiver de las calles de Madrid.
    Raises:
        ServiceNotAvailableError: Si no es posible recuperar el grafo de OpenStreetMap.
    """
    file = "madrid.graphml"

    if os.path.exists(file):
        print("Cargando grafo desde fichero...")
        G = ox.load_graphml(file)
    else:
        print("Descargando grafo de OSM (esto puede tardar)...")
        G = ox.graph_from_place("Madrid, Spain", network_type="drive")
        ox.save_graphml(G, file)
        print(f"Grafo guardado como {file}")

    return procesa_grafo(G)

def dibujar_grafo_osmnx(G):
    pos = {n: (G.nodes[n]["x"], G.nodes[n]["y"]) for n in G.nodes}

    print("Dibujando grafo procesado...")
    plt.figure(figsize=(12,12))

    nx.draw(G, pos=pos, node_size=1, width=0.1, edge_color="gray", arrows=True)

    plt.show()

def procesa_grafo(multidigrafo:nx.MultiDiGraph) -> nx.DiGraph:
    """ Función que recupera el quiver de calles de Madrid de OpenStreetMap.
    Args:
        multidigrafo: multidigrafo de las calles de Madrid obtenido de OpenStreetMap.
    Returns:
        nx.DiGraph: Grafo dirigido y sin bucles asociado al multidigrafo dado.
    Raises: None
    """
    G = ox.convert.to_digraph(multidigrafo)
    loops = list(nx.selfloop_edges(G))
    if loops:
        G.remove_edges_from(loops)
    return G

G= carga_grafo()
print(type(G))

In [ ]:
dibujar_grafo_osmnx(G)

In [ ]:
print(G.nodes)
for node in G.nodes:
    print(G[node])


In [ ]:
def obtener_datos_arista(G: nx.Graph, u: object, v: object) -> dict:
    """
    Devuelve el diccionario de atributos de la arista (u, v) tanto
    para Graph/DiGraph como para MultiGraph/MultiDiGraph.
    """
    data = G.get_edge_data(u, v)
    if data is None:
        return {}

    # MultiGraph / MultiDiGraph: data es un dict de dicts (key -> atributos)
    if isinstance(G, (nx.MultiGraph, nx.MultiDiGraph)):
        # Elegimos la arista con menor longitud (o cualquiera si no hay)
        best_key = None
        best_data = None
        best_length = float("inf")
        for k, d in data.items():
            length = d.get("length", float("inf"))
            if length < best_length:
                best_length = length
                best_key = k
                best_data = d
        return best_data if best_data is not None else {}
    else:
        # Graph / DiGraph: data ya es un dict de atributos
        return data

In [ ]:
obtener_datos_arista(G, 171946, 171954)